# Home work  -> create a View using - transform function - which returns transformed_df

In [0]:
# first call the function then save the output in the df

import pyspark.sql.functions as F
from pyspark.sql.functions import *

# df is already read from cells below - so not run here again

transformed_df = transform_customer(df)
transformed_df.createOrReplaceTempView('View_cleaned_customer_df')

transformed_df.show()

# if we want to do it in sql we can use this but silver schema is not ready
#spark.sql("CREATE OR REPLACE VIEW retaildataplatform.silver.final_transformed_customer AS SELECT * FROM View_cleaned_customer_df " )


In [0]:

# step 1 - read the customer table from bronze
# step 2 - display the dataframe n try to identify what cleaning can be done
# step 3 - Apply the required transformation steps which u identified n then store in dataframe
# step 4 - in final dataframe we will apply SCD - SCD type1 and then store in silver schema (before that silver schema using SQL)
# step 5 - write a function in same notebook which clean n transform customers data from bronze and returns the dataframe 
# step 6 - write a function which can do scd1 - if we provide the parameters like source_dataframe, buisness key, target_table [ after creating this function scd1 we will take it to common_utils and then reuse it here and in other notebooks]
# step 7 - take out all hardcoded values i.e full moduralize the code ( hardcoded values to json config, proper logging setup n all reusable functions should have used)

In [0]:
#step 1
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")
df.show()

In [0]:
# column transformation - after looking at data customer data - we can see smaller case customer_name, it might come from client
import pyspark.sql.functions as F
from pyspark.sql.functions import *

df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")


def transform_customer(df):

    df = df.withColumn('customer_name',F.lower(df['customer_name']))\
    .withColumn('name_parts',F.split(F.col("customer_name"),","))\
    .withColumn('first_name',F.expr('get(name_parts,1)'))\
    .withColumn('last_name',F.expr('get(name_parts,0)'))\
    .drop('name_parts')\
    .drop('customer_name')\
        .withColumn('city',F.upper(df['city']))\
        .withColumn('postcode', regexp_replace(df['postcode'], r'\.0$', ''))\
            .drop('file_path')\
            .withColumn("valid_from", from_unixtime(expr("try_cast(valid_from as bigint)"))) \
             .withColumn("valid_to", from_unixtime(expr("try_cast(valid_to as bigint)")))\
                 .withColumn('country',lit('USA'))\
                     .dropDuplicates(['customer_id'])


    return df.select(['customer_id',
                            'first_name',
            'last_name',
            'tax_id',
            'tax_code',
            'state',
            'city',
            'country',
            'postcode',
            'street',
            'number',
            'unit',
            'region',
            'district',
            'lon',
            'lat',
            'ship_to_address',
            'valid_from',
            'valid_to',
            'units_purchased',
            'loyalty_segment',
            'last_update_ts'
            ])




In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import *

df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")


cleaned_customer = transform_customer(df)


cleaned_customer.show()


# step 5 - SCD1  -update - changed rows and insert new rows

In [0]:
# we need buisness key or unique identifier - to check particular record changed or not - check whether exist or not

# we dont have no silver schema no customer table
# CHECK if table exist or not ----- if exist then do scd1 or do full load


In [0]:
source_df = df
target_df = ?

In [0]:


if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print('table exists proceed with SCD1')
else:
    print('no table exists do full load')
    spark.sql("""CREATE Schema if not exists silver""")
    df.write.format('delta').mode('overwrite').saveAsTable('retaildataplatform.silver.customers')
    # do scd1
    # we will write a function which will do scd1
    # we will take this function to common_utils
    # we will use this